# Coffee17 Factorized Bilinear Conv — konfirmasi tiga seed
Validation-only: memakai ulang seed 42 dan melatih seed 123/2026. Test tetap terkunci.

In [ ]:
SEEDS = [42, 123, 2026]
REPO_REF = 'agent/hong-classification-ablation'
HF_REPO = 'ediprin/coffee-backbone-checkpoints'
DRIVE_FOLDER = 'coffee17-factorized-bilinear-conv'
EVALUATION_SPLIT = 'val'

In [ ]:
# 1/5 SETUP REPO, DRIVE, GPU, DAN DATASET
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
assert SEEDS == [42, 123, 2026] and EVALUATION_SPLIT == 'val'
repo = Path('/content/coffee-bean-classification')
repo_url = 'https://github.com/ediprin/coffee-bean-classification.git'
def run(command, cwd=None):
    command = [str(item) for item in command]
    print('\n$', ' '.join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)
if (repo / '.git').is_dir():
    run(['git', 'fetch', 'origin', REPO_REF], cwd=repo)
    run(['git', 'checkout', REPO_REF], cwd=repo)
    run(['git', 'pull', '--ff-only', 'origin', REPO_REF], cwd=repo)
else:
    if repo.exists(): shutil.rmtree(repo)
    run(['git', 'clone', '--branch', REPO_REF, repo_url, repo])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', repo])
import torch
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
dataset_base = Path('/content/coffee17-fbconv-data')
original, clean = dataset_base / 'original', dataset_base / 'clean'
archive = dataset_base / 'coffee17.zip'
data_root = clean / 'folds/fold_1'
suffixes = {'.jpg', '.jpeg', '.png'}
def image_count(path):
    return sum(1 for p in path.rglob('*') if p.is_file() and p.suffix.lower() in suffixes) if path.is_dir() else 0
if image_count(original / 'source') != 979:
    if original.exists(): shutil.rmtree(original)
    if archive.exists() and not zipfile.is_zipfile(archive): archive.unlink()
    run([sys.executable, '-u', '-m', 'bilinear_lmmd.data.preparation.prepare_coffee17', '--output', original, '--archive', archive, '--seed', '42'], cwd=repo)
if not (clean / 'audit.json').is_file():
    if clean.exists(): shutil.rmtree(clean)
    run([sys.executable, '-u', '-m', 'bilinear_lmmd.data.preparation.prepare_clean_grouped_folds', '--source-root', original / 'source', '--output-root', clean, '--expected-count', '979', '--folds', '5', '--seed', '42', '--validation-ratio', '0.1'], cwd=repo)
assert all((data_root / f'source/{split}').is_dir() for split in ('train', 'val', 'test'))
output_root = Path('/content/drive/MyDrive') / DRIVE_FOLDER
output_root.mkdir(parents=True, exist_ok=True)
print('DATASET:', data_root)
print('OUTPUT :', output_root)

In [ ]:
# 2/5 PASTIKAN SCREENING SEED 42 SUDAH PASS
screening_path = output_root / 'val_reports/fbconv_screening_decision.json'
assert screening_path.is_file(), 'Report screening seed 42 tidak ada. Jalankan notebook screening terlebih dahulu.'
screening = json.loads(screening_path.read_text())
assert screening.get('seeds') == [42] and screening.get('final_decision') == 'PASS', 'Screening seed 42 belum PASS.'
print('SCREENING:', screening['final_decision'])

In [ ]:
# 3/5 RESTORE / LENGKAPI BASELINE TIGA SEED
token = userdata.get('HF_TOKEN')
assert token, 'Tambahkan HF_TOKEN ke Colab Secrets.'
os.environ['HF_TOKEN'] = token
from huggingface_hub import snapshot_download
baseline_root = Path('/content/backbone-results')
patterns = [f'outputs/{code}_seed{seed}/*' for code in ('BE2G', 'BE2H') for seed in SEEDS]
snapshot_download(repo_id=HF_REPO, repo_type='model', token=token, local_dir=baseline_root, allow_patterns=patterns)
command = [sys.executable, '-u', '-m', 'bilinear_lmmd.experiments.run_backbone_screening', '--data-root', str(data_root), '--output-root', str(baseline_root), '--seeds', *map(str, SEEDS), '--backbones', 'EV2', '--heads', 'gap', 'hbp', '--evaluation-split', 'val', '--hf-repo', HF_REPO, '--hf-sync-every', '2']
process = subprocess.Popen(command, cwd=repo)
while process.poll() is None:
    print('[BASELINE] restore/evaluasi masih berjalan', flush=True)
    time.sleep(60)
assert process.wait() == 0, 'Baseline gagal; lihat error di atas.'

In [ ]:
# 4/5 TRAIN / RESUME FB0 DAN FB1; SEED 42 OTOMATIS DILEWATI
command = [sys.executable, '-u', '-m', 'bilinear_lmmd.experiments.run_factorized_bilinear_conv_confirmation', '--data-root', str(data_root), '--baseline-root', str(baseline_root), '--output-root', str(output_root), '--seeds', *map(str, SEEDS), '--evaluation-split', EVALUATION_SPLIT]
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.Popen(command, cwd=repo)
started = time.monotonic()
while process.poll() is None:
    rows = []
    for code in ('FB0', 'FB1'):
        for seed in SEEDS:
            history = output_root / f'outputs/{code}_seed{seed}/history.json'
            if history.is_file():
                try: rows.append(f'{code}-{seed}={len(json.loads(history.read_text()))}/50')
                except Exception: rows.append(f'{code}-{seed}=saving')
    print(f'[KONFIRMASI {(time.monotonic()-started)/60:.1f} menit]', ', '.join(rows) if rows else 'inisialisasi', flush=True)
    time.sleep(60)
assert process.wait() == 0, 'Konfirmasi gagal; lihat error di atas.'

In [ ]:
# 5/5 TAMPILKAN AGREGAT TIGA SEED DAN PUTUSAN
path = output_root / 'val_reports/fbconv_confirmation.json'
assert path.is_file(), f'Belum ada: {path}'
report = json.loads(path.read_text())
for name, summary in report['comparisons'].items():
    print(f'\n=== {name} ===')
    for key in ('macro_f1', 'hard_class_f1', 'worst_class_f1'):
        row = summary[key]
        print(f"{key:22s}: {row['baseline_mean']:.2%} -> {row['candidate_mean']:.2%} Delta={row['delta_mean']:+.2%} +/- {row['delta_std']:.2%} naik={row['improved_seeds']}/3")
for name, row in report['decisions'].items(): print(name, row['decision'], row['criteria'])
print('FINAL:', report['final_decision'])
print('Test dibuka:', report['test_opened'])

## Aturan keputusan
`FINAL: PASS` memerlukan FB1 mengalahkan GAP dan FB0 pada rata-rata Macro/Hard, menang minimal 2/3 seed, dan menjaga rata-rata Worst-F1 dalam toleransi -1 poin. Test tetap tidak dibuka pada notebook ini. Bila runtime reset, jalankan ulang semua cell; hasil FB0/FB1 tersimpan langsung di Google Drive.